# Notebook 01 — Data Preparation

**Goal:** Convert raw surah-level MP3 files (Ali al-Hudhaifi, Qaloon) into
individual ayah-level 16 kHz mono WAV segments that Wav2Vec2 can consume.

**Why this notebook exists:**
The base model requires 16 kHz mono audio.  Our source files are surah-level
MP3s — one file per chapter, each 2–60 minutes long.  We must split each file
into one segment per ayah and normalize loudness so amplitude variation doesn't
destabilize training.  All heavy I/O runs here once; downstream notebooks only
read the processed WAVs.

**Output:** `data/processed/segments/<surah>_<ayah>.wav`

## Step 0 — Install and import dependencies

We install from `requirements.txt` so the Colab runtime matches the local
development environment exactly.

In [ ]:
# Mount Google Drive so we can read audio files and write results.
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/drive/MyDrive/iqraa-ai')

!pip install -q -r /content/drive/MyDrive/iqraa-ai/requirements.txt

## Step 1 — Configure paths

All paths are defined in one cell so they are easy to change without hunting
through the notebook.

In [ ]:
from pathlib import Path

REPO_ROOT    = Path('/content/drive/MyDrive/iqraa-ai')
AUDIO_DIR    = REPO_ROOT / 'data' / 'audio'
SEGMENTS_DIR = REPO_ROOT / 'data' / 'processed' / 'segments'
TEXT_FILE    = REPO_ROOT / 'data' / 'text' / 'qalon_canonical.json'

SEGMENTS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Audio source : {AUDIO_DIR}')
print(f'Output dir   : {SEGMENTS_DIR}')

## Step 2 — Load canonical text for segment boundaries

We need the text file here (not for labels yet, but to know how many ayahs
each surah has so we can validate the number of segments we produce).

In [ ]:
import json
from collections import defaultdict

with open(TEXT_FILE, encoding='utf-8') as f:
    ayahs = json.load(f)

ayahs_per_surah = defaultdict(list)
for a in ayahs:
    ayahs_per_surah[a['surah']].append(a)

print(f'Total ayahs: {len(ayahs)}')
print(f'Total surahs: {len(ayahs_per_surah)}')

## Step 3 — Load, resample, and loudness-normalize each surah

We convert to 16 kHz mono (Wav2Vec2 requirement) and normalize to -23 LUFS
(EBU R128 broadcast standard) so all segments have comparable amplitude.
This reduces training instability caused by loudness variation across surahs.

In [ ]:
import librosa
import numpy as np
import pyloudnorm as pyln
import soundfile as sf
from tqdm import tqdm

TARGET_SR   = 16_000
TARGET_LUFS = -23.0
meter       = pyln.Meter(TARGET_SR)

def load_and_normalize(mp3_path: Path) -> np.ndarray:
    audio, sr = librosa.load(str(mp3_path), sr=TARGET_SR, mono=True)
    loudness  = meter.integrated_loudness(audio)
    audio     = pyln.normalize.loudness(audio, loudness, TARGET_LUFS)
    return audio

# Verify loading works on the first surah
test_mp3 = next(AUDIO_DIR.glob('*.mp3'), None)
if test_mp3:
    sample = load_and_normalize(test_mp3)
    print(f'Loaded {test_mp3.name}: {len(sample)/TARGET_SR:.1f}s')
else:
    print('No MP3 files found — place them in data/audio/')

## Step 4 — Segment surah audio into ayah-level clips

**TODO (Notebook 03 will refine this):** This cell uses a simple energy-based
silence detector as a placeholder.  Notebook 03 will replace it with
forced-alignment using the canonical text, which gives more accurate boundaries.

For now we split on silence gaps >300 ms and assign segments to ayahs in order.

In [ ]:
import librosa

def segment_by_silence(audio: np.ndarray, sr: int,
                        min_silence_ms: int = 300,
                        top_db: int = 30) -> list[np.ndarray]:
    """Split audio on silence gaps, return list of non-silent clips."""
    intervals = librosa.effects.split(audio, top_db=top_db,
                                      frame_length=1024, hop_length=256)
    min_gap_samples = int(min_silence_ms / 1000 * sr)
    segments = []
    for start, end in intervals:
        if end - start > min_gap_samples:
            segments.append(audio[start:end])
    return segments

print('segment_by_silence defined — run the full loop in the next cell')

## Step 5 — Process all surahs and save WAV segments

For each surah MP3 we: load → normalize → segment → save individual WAV files.
Files are named `<surah_padded>_<ayah_padded>.wav` (e.g. `001_001.wav`).

In [ ]:
mp3_files = sorted(AUDIO_DIR.glob('*.mp3'))
print(f'Found {len(mp3_files)} MP3 files')

for mp3_path in tqdm(mp3_files, desc='Surahs'):
    surah_num = int(mp3_path.stem)  # expects "001.mp3" naming
    expected_ayahs = len(ayahs_per_surah[surah_num])

    audio    = load_and_normalize(mp3_path)
    segments = segment_by_silence(audio, TARGET_SR)

    if len(segments) != expected_ayahs:
        print(f'  WARNING surah {surah_num}: got {len(segments)} segments, '
              f'expected {expected_ayahs}')

    for ayah_idx, seg in enumerate(segments, start=1):
        out_path = SEGMENTS_DIR / f'{surah_num:03d}_{ayah_idx:03d}.wav'
        sf.write(str(out_path), seg, TARGET_SR)

print('Done.')